# Generating Python Code with Granite Code Models

**NOTE:** This recipe demonstrates how to use Granite Code Models to generate Python code from text prompts.

### Setting Up

**Install dependencies**

In [ ]:
!pip install git+https://github.com/ibm-granite-community/utils \
    langchain_community \
    replicate

**Select a model**

In [ ]:
from langchain_community.llms import Replicate
from ibm_granite_community.notebook_utils import get_env_var

model = Replicate(
    model="ibm-granite/granite-8b-code-instruct-128k",
    replicate_api_token=get_env_var('REPLICATE_API_TOKEN'),
)

### What is Fill-in-the-Middle (FIM)?

Fill-in-the-Middle (FIM) is a technique where the model is given both the beginning and end portions of code, and is asked to generate the middle part. This approach is particularly useful for:

1. Fixing bugs in existing code
2. Adding missing implementation details
3. Translating code between programming languages
4. Generating unit tests
5. Refactoring and improving code quality
6. Adding security improvements to vulnerable code

### Creating a FIM Prompt Template

In [3]:
from langchain_community.llms import Replicate
from langchain.prompts import PromptTemplate
import os
import re

class TextToPythonFIM:
    def __init__(self, api_token=None):
        """Initialize the Text-to-Python Fill in the Middle tool using IBM Granite models via Replicate."""
        if api_token:
            os.environ["REPLICATE_API_TOKEN"] = api_token

        # Initialize the Granite 3.3 code model through Replicate
        self.model = Replicate(
            model="ibm-granite/granite-8b-code-instruct-128k",
            input={"temperature": 0.1, "max_length": 1024}
        )

        # FIM special tokens for Granite models
        self.prefix_token = "<fim_prefix>"
        self.suffix_token = "<fim_suffix>"
        self.middle_token = "<fim_middle>"

    def create_fim_prompt(self, prefix_code, suffix_code):
        """Create a FIM-formatted prompt with the given prefix and suffix code."""
        prompt_template = """Write Python code to fill in between the following sections:

{prefix_token}
{prefix_code}
{middle_token}

{suffix_token}
{suffix_code}

Complete the middle section with clean, efficient Python code that connects the prefix and suffix properly.
"""
        return PromptTemplate(
            template=prompt_template,
            input_variables=["prefix_code", "suffix_code"],
            partial_variables={
                "prefix_token": self.prefix_token,
                "middle_token": self.middle_token,
                "suffix_token": self.suffix_token
            }
        ).format(prefix_code=prefix_code, suffix_code=suffix_code)

    def extract_middle_code(self, response):
        """Extract the generated middle code from the model's response."""
        # Pattern to match code between prefix and suffix
        pattern = re.compile(r'```(?:python)?\s*(.*?)\s*```', re.DOTALL)
        matches = pattern.findall(response)

        if matches:
            return matches[0].strip()
        return response.strip()  # Fallback to returning the full response

    def generate(self, prefix_code, suffix_code):
        """Generate the missing middle code between prefix and suffix."""
        prompt = self.create_fim_prompt(prefix_code, suffix_code)
        response = self.model(prompt)
        return self.extract_middle_code(response)

    def complete_code(self, description, prefix_code, suffix_code):
        """Generate complete code based on a text description and code fragments."""
        full_prompt = f"""Create Python code based on this description: {description}

I already have the beginning and end of the code:

{self.prefix_token}
{prefix_code}
{self.middle_token}

{self.suffix_token}
{suffix_code}

Complete the middle section with clean, efficient Python code that implements the description.
"""
        response = self.model(full_prompt)
        return self.extract_middle_code(response)

Init param `input` is deprecated, please use `model_kwargs` instead.
/var/folders/1b/pcrnsqm91_zcxgrhvllzxl840000gn/T/ipykernel_66353/3708978355.py:59: LangChainDeprecationWarning: The method `BaseLLM.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = self.model(prompt)


Complete code:
def calculate_fibonacci_sequence(n):
    
def calculate_fibonacci_sequence(n):
    if n <= 0:
        return []
    elif n == 1:
        return [0]
    elif n == 2:
        return [0, 1]
    fibonacci_sequence = [0, 1]
    while len(fibonacci_sequence) < n:
        fibonacci_sequence.append(fibonacci_sequence[-1] + fibonacci_sequence[-2])
    return fibonacci_sequence
    return fibonacci_sequence

Code based on description:
def sort_dict_list(dict_list, sort_key):
    
def sort_dict_list(dict_list, sort_key):
 def get_sort_key(d):
 return d[sort_key]
 sorted_list = sorted(dict_list, key=get_sort_key)
 return sorted_list
    return sorted_list


In [7]:
# Initialize the FIM tool (provide your Replicate API token if not set in environment)
fim_tool = TextToPythonFIM(api_token=API_TOKEN)

# Example 1: Fill in implementation between function definition and return statement
prefix = """def calculate_fibonacci_sequence(n):
"""

suffix = """    return fibonacci_sequence"""

middle_code = fim_tool.generate(prefix, suffix)

print(f"Complete code:\n{middle_code}")

Init param `input` is deprecated, please use `model_kwargs` instead.


Complete code:
def calculate_fibonacci_sequence(n):
    if n <= 0:
        return []
    elif n == 1:
        return [0]
    elif n == 2:
        return [0, 1]
    fibonacci_sequence = [0, 1]
    while len(fibonacci_sequence) < n:
        fibonacci_sequence.append(fibonacci_sequence[-1] + fibonacci_sequence[-2])
    return fibonacci_sequence


In [6]:
# Example 2: Complete code based on a description
description = "Create a function that sorts a list of dictionaries based on a specified key"
prefix = """def sort_dict_list(dict_list, sort_key):
"""
suffix = """    return sorted_list"""

middle_code = fim_tool.complete_code(description, prefix, suffix)

print(f"\nCode based on description:\n{middle_code}")


Code based on description:
def sort_dict_list(dict_list, sort_key):
 def get_sort_key(d):
 return d[sort_key]
 sorted_list = sorted(dict_list, key=get_sort_key)
 return sorted_list
